# Bootstrap-phase analysis — how chento turned $200 into something meaningful

Educational deep-dive into the first ~6 months of the chento journal
(June 2024 → November 2024). This is the **100-1k Challenge era** — the
phase he explicitly retired in Nov 2024 (*"End of an era. Hereby stopping
the 100-1k challenge for good"*) and later refined out of his Jan 2026
$200k→$2M rulebook (*"adding to losers banned"*).

**Why study it anyway**: he ran this style **19 successful times** by his
own count before the first failure (per his June 10 2024 introduction
post). Whatever's in here, the technique works for compounding a small
account fast. Not bot-viable as-is (200x leverage scalping needs real-time
tape reading), but understanding the moves helps us identify which parts
ARE codifiable.

## Data sources

- `studies/material/chento/phase1_trades.jsonl` — 29 extracted trades + balance screenshots
- `studies/material/chento/messages.jsonl` — text + image manifest
- `data/databases/prod.db.op_perp_1m` — Binance OPUSDT perp 1m bars (1.99M rows)
- `data/databases/prod.db.btc_1m` — BTC 1m

## What we'll answer

1. **Equity curve reconstruction** — when did the bankroll grow, when did it dip?
2. **Asset mix** — OP-dominant? Confirmed. How much of growth came from OP?
3. **Leverage progression** — how did he scale leverage as the bankroll grew?
4. **The big winners** — which trades drove the bulk of compound returns?
5. **The DCA-up pattern** — how often did he average UP into winners?
6. **Hold times** — minutes? Hours? Days?
7. **Position-sizing-relative-to-equity** — was his risk-per-trade actually disciplined?
8. **What's codifiable** — which behaviors could a bot replicate, which couldn't?

In [ ]:
import sys, json, sqlite3
from pathlib import Path
from datetime import datetime, timedelta, timezone
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

ROOT = Path.cwd()
while not (ROOT / 'data' / 'databases' / 'prod.db').exists():
    if ROOT == ROOT.parent: raise RuntimeError('could not locate prod.db')
    ROOT = ROOT.parent
DB = ROOT / 'data' / 'databases' / 'prod.db'
MSGS = ROOT / 'studies' / 'material' / 'chento' / 'messages.jsonl'
TRADES = ROOT / 'studies' / 'material' / 'chento' / 'phase1_trades.jsonl'

trades = [json.loads(l) for l in TRADES.read_text(encoding='utf-8').splitlines() if l.strip()]
print(f'phase1_trades.jsonl: {len(trades)} records')
print(f'screenshot_type counts:')
for t, n in Counter(r.get('screenshot_type') for r in trades).most_common():
    print(f'  {t}: {n}')

## 1. Equity-curve reconstruction

He posted occasional balance screenshots; combined with the chronology of
his trade captions (*"Port 130"*, *"DROPS THE FKING MIC"* etc.), we can
reconstruct a rough equity curve.

Sources:
1. Explicit balance screenshots (3 in phase1_trades.jsonl)
2. Text captions mentioning portfolio value (regex on `messages.jsonl`)
3. Implied moves from trade outcomes (where the screenshot shows margin + PnL)

In [ ]:
import re
msgs = [json.loads(l) for l in MSGS.read_text(encoding='utf-8').splitlines() if l.strip()]
chento = [r for r in msgs if r['author_id'] == '978925049945919499']

# Explicit balance screenshots
balance_screens = [r for r in trades if r.get('screenshot_type') == 'account_balance']
print(f'Balance screenshots ({len(balance_screens)}):')
for r in balance_screens:
    print(f"  {r['ts'][:16]}  ${r.get('total_assets_usdt', 0):,.0f}  (futures ${r.get('futures_value_usdt', 0):,.0f})")

# Text mentions of port values
port_re = re.compile(r'port\s*[~$]?\s*(\d{2,5})(?!\d)', re.I)
milestone_re = re.compile(r'(?:closing|closed|profit|made|hit)\s*[~$]?\s*(\d{1,2}[\.,]?\d{0,3})\s*[kK]?', re.I)
port_mentions = []
for r in chento:
    for m in port_re.finditer(r['text']):
        port_mentions.append({'ts': r['ts_utc'], 'value': int(m.group(1)),
                              'text': r['text'][:80]})
port_mentions.sort(key=lambda x: x['ts'])
print(f'\nPort-value mentions in text ({len(port_mentions)}):')
for pm in port_mentions:
    print(f"  {pm['ts'][:16]}  $~{pm['value']:>5}  text={pm['text']!r}")

In [ ]:
# Build the equity-curve dataframe combining both sources
rows = [{'ts': datetime.fromisoformat(r['ts']), 'equity': r['total_assets_usdt'],
          'source': 'balance_screen'} for r in balance_screens]
for pm in port_mentions:
    rows.append({'ts': datetime.fromisoformat(pm['ts']), 'equity': pm['value'],
                  'source': 'caption'})
eq = pd.DataFrame(rows).sort_values('ts').reset_index(drop=True)
eq = eq[eq['equity'] > 30]  # drop obviously-noisy mentions (e.g. "20$" referring to stop dollars)
print('Reconstructed equity curve:')
print(eq.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
for src, grp in eq.groupby('source'):
    ax.scatter(grp['ts'], grp['equity'], s=60,
               label=src, alpha=0.85)
ax.plot(eq['ts'], eq['equity'], color='cyan', lw=1.0, alpha=0.6)
ax.set_yscale('log')
ax.set_title('chento bootstrap equity curve (log scale) — June–Sept 2024')
ax.set_ylabel('account equity (USDT, log)'); ax.set_xlabel('date')
ax.axhline(200, color='gray', lw=0.5, ls='--', label='start: $200')
ax.axhline(1000, color='gray', lw=0.3, ls=':')
ax.axhline(5000, color='gray', lw=0.3, ls=':')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

## 2. Asset mix in trades extracted

Confirmed: OP was the bootstrap engine.

In [ ]:
card_trades = [r for r in trades if r.get('screenshot_type') == 'exchange_position_card']
print(f'Position-card trades: {len(card_trades)}')

asset_dir = Counter()
for r in card_trades:
    asset_dir[(r['asset'], r['direction'])] += 1
print('\nasset × direction (unique trades by tuple):')
for (a, d), n in asset_dir.most_common():
    print(f'  {a:10s} {d:6s}: {n}')

# Aggregate per asset
by_asset = Counter(r['asset'] for r in card_trades)
print('\nby asset:')
for a, n in by_asset.most_common():
    print(f'  {a:10s}: {n} trades ({n/len(card_trades)*100:.0f}%)')

## 3. Leverage progression

How did he ramp leverage as the bankroll grew?

In [ ]:
lev_df = pd.DataFrame([{
    'ts': datetime.fromisoformat(r['ts']),
    'asset': r['asset'], 'direction': r['direction'],
    'leverage': r['leverage'],
    'margin_usdt': r['margin_usdt'],
    'position_size_usdt': r['position_size_usdt'],
} for r in card_trades])
lev_df = lev_df.sort_values('ts').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 4.5))
colors = {'long': 'lime', 'short': 'orangered'}
for direction in ['long', 'short']:
    sub = lev_df[lev_df['direction'] == direction]
    ax.scatter(sub['ts'], sub['leverage'], s=80,
               c=colors[direction], label=direction, alpha=0.7)
ax.set_ylabel('leverage (x)'); ax.set_xlabel('date')
ax.set_title('Leverage per trade — bootstrap era (June–Sept 2024)')
ax.legend(loc='upper left')
ax.grid(alpha=0.2)
for ts_label, lev_label in [('2024-07-16', 50), ('2024-08-27', 49), ('2024-09-27', 125)]:
    ax.annotate(f'{lev_label}x', xy=(datetime.fromisoformat(ts_label+'T12:00'), lev_label),
                xytext=(0, 10), textcoords='offset points', ha='center', fontsize=8, color='cyan')
plt.tight_layout(); plt.show()

print(f'\nLeverage stats:')
print(f'  median: {lev_df["leverage"].median():.0f}x')
print(f'  mean:   {lev_df["leverage"].mean():.1f}x')
print(f'  max:    {lev_df["leverage"].max():.0f}x')
print(f'  distribution: {dict(Counter(lev_df["leverage"]))}')

## 4. Big winners — which trades carried the compound returns?

From the extracted trades, find the moments with the biggest PnL contribution.

In [ ]:
pnl_rows = []
for r in card_trades:
    pnl = r.get('unrealized_pnl_usdt') or 0
    margin = r.get('margin_usdt') or 0
    pnl_pct = r.get('unrealized_pnl_pct') or 0
    pnl_rows.append({
        'ts': datetime.fromisoformat(r['ts']),
        'asset': r['asset'], 'direction': r['direction'],
        'leverage': r['leverage'],
        'margin': margin, 'pnl_usdt': pnl, 'pnl_pct': pnl_pct,
        'notes': r.get('notes', '')[:100],
    })
pnl_df = pd.DataFrame(pnl_rows).sort_values('pnl_usdt', ascending=False)
print('Top 10 trade screenshots by unrealized PnL ($):')
print(pnl_df.head(10)[['ts','asset','direction','leverage','margin','pnl_usdt','pnl_pct']].to_string(index=False))

print('\nTop 5 by % return on margin:')
print(pnl_df.sort_values('pnl_pct', ascending=False).head(5)[['ts','asset','direction','leverage','margin','pnl_usdt','pnl_pct']].to_string(index=False))

## 5. The DCA-up pattern

He averaged INTO winners. Let's quantify by finding consecutive trades on
the same asset where the entry price moved (= DCA event).

In [ ]:
# Group by (asset, direction) and look at consecutive entries with shifting entry prices
trades_sorted = sorted(card_trades, key=lambda r: r['ts'])
dca_events = []
prev = {}  # key=(asset,direction) -> last trade record
for r in trades_sorted:
    key = (r['asset'], r['direction'])
    last = prev.get(key)
    if last:
        gap_h = (datetime.fromisoformat(r['ts']) - datetime.fromisoformat(last['ts'])).total_seconds()/3600
        entry_change = (r['entry_price'] - last['entry_price'])/last['entry_price']*100
        margin_change = (r['margin_usdt'] - last['margin_usdt'])/max(last['margin_usdt'],1e-9)*100
        if abs(entry_change) > 0.1 and gap_h < 24:  # within a day + entry moved
            dca_events.append({
                'ts': r['ts'][:16],
                'asset': r['asset'], 'direction': r['direction'],
                'gap_h': round(gap_h, 1),
                'prev_entry': last['entry_price'],
                'new_entry': r['entry_price'],
                'entry_change_pct': round(entry_change, 2),
                'prev_margin': last['margin_usdt'],
                'new_margin': r['margin_usdt'],
                'margin_change_pct': round(margin_change, 0),
                'dca_direction': 'UP' if entry_change > 0 else 'DOWN',
            })
    prev[key] = r

if dca_events:
    dca_df = pd.DataFrame(dca_events)
    print(f'DCA events detected: {len(dca_df)}')
    print(f'  UP (avg into winners): {(dca_df["dca_direction"]=="UP").sum()}')
    print(f'  DOWN (avg into losers): {(dca_df["dca_direction"]=="DOWN").sum()}')
    print()
    print(dca_df.to_string(index=False))
else:
    print('No DCA events detected — sample too sparse')

## 6. Hold times via OP price action

We have OPUSDT 1m perp data — we can verify approx hold times of his major
trades by tracking when his entry price would have been touched and when
the move he benefited from happened.

In [ ]:
con = sqlite3.connect(str(DB))
# Just verify OP coverage matches phase1 trade dates
r = con.execute('SELECT MIN(open_time), MAX(open_time), COUNT(*) FROM op_perp_1m').fetchone()
con.close()
print(f'op_perp_1m: {r[2]:,} rows')
print(f'  first: {datetime.fromtimestamp(r[0]/1000, tz=timezone.utc)}')
print(f'  last:  {datetime.fromtimestamp(r[1]/1000, tz=timezone.utc)}')

# For each OP trade in our ledger, look up what OP was doing around it
op_trades = [r for r in card_trades if r['asset'] == 'OPUSDT']
print(f'\n{len(op_trades)} OPUSDT trade screenshots in ledger')

# For the famous 2024-06-18 "DROPS THE FKING MIC" trade — we already validated
# this in validate_chento_trade.py. Repeat the headline number:
famous = next((r for r in op_trades if r['ts'].startswith('2024-06-18')), None)
if famous:
    print(f"\nThe famous trade ({famous['ts'][:16]}):")
    print(f"  entry: {famous['entry_price']}, mark: {famous['mark_price']}")
    print(f"  PnL: ${famous['unrealized_pnl_usdt']:,.2f} ({famous['unrealized_pnl_pct']:+.1f}%)")
    print(f"  notes: {famous.get('notes', '')[:200]}")

In [ ]:
# Visualize OP price action through the bootstrap window
con = sqlite3.connect(str(DB))
start_ms = int(datetime(2024, 6, 1, tzinfo=timezone.utc).timestamp() * 1000)
end_ms = int(datetime(2024, 10, 1, tzinfo=timezone.utc).timestamp() * 1000)
op_df = pd.read_sql(
    'SELECT open_time, open, high, low, close FROM op_perp_1m '
    'WHERE open_time BETWEEN ? AND ? ORDER BY open_time',
    con, params=(start_ms, end_ms))
con.close()
op_df['ts'] = pd.to_datetime(op_df['open_time'], unit='ms', utc=True)
op_df = op_df.set_index('ts').drop(columns='open_time')
# Resample to 1h for visualization
op_1h = op_df.resample('1h').agg(o=('open','first'), h=('high','max'),
                                 l=('low','min'), c=('close','last')).dropna()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(op_1h.index, op_1h['c'], color='cyan', lw=0.8, label='OP price (1h close)')
# Overlay his trade screenshots as markers
for r in op_trades:
    ts = datetime.fromisoformat(r['ts'])
    px = r['mark_price']
    color = 'lime' if r['direction'] == 'long' else 'orangered'
    ax.scatter(ts, px, s=40, c=color, marker='v' if r['direction']=='short' else '^',
               edgecolors='white', linewidth=0.5, zorder=5)
ax.set_title('OPUSDT 2024-06 to 2024-10 — chento bootstrap trades overlaid')
ax.set_ylabel('OP / USDT'); ax.set_xlabel('date')
ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print(f'OP range June-Sept 2024: ${op_1h["l"].min():.4f} → ${op_1h["h"].max():.4f}')
print(f'Drawdown peak→trough: {(op_1h["l"].min()/op_1h["h"].max() - 1) * 100:.1f}%')

## 7. Position-size relative to equity

Was his actual risk-per-trade disciplined? The bootstrap-era captions said
*"Sl 20$"* and *"Risking 20$"* — let's check if margin allocation was
consistent with those captions.

In [ ]:
# Use the balance screenshots to interpolate equity at each trade time
balance_df = pd.DataFrame([{
    'ts': datetime.fromisoformat(r['ts']),
    'equity': r['total_assets_usdt']
} for r in balance_screens]).sort_values('ts')

def equity_at(ts):
    """Linear-interpolate equity from the balance-screen anchors."""
    if balance_df.empty: return None
    after = balance_df[balance_df['ts'] >= ts]
    before = balance_df[balance_df['ts'] <= ts]
    if before.empty: return float(after.iloc[0]['equity'])
    if after.empty: return float(before.iloc[-1]['equity'])
    eq_before, t_before = float(before.iloc[-1]['equity']), before.iloc[-1]['ts']
    eq_after, t_after = float(after.iloc[0]['equity']), after.iloc[0]['ts']
    if t_after == t_before: return eq_before
    frac = (ts - t_before).total_seconds() / max((t_after - t_before).total_seconds(), 1)
    return eq_before + frac * (eq_after - eq_before)

size_rows = []
for r in card_trades:
    ts = datetime.fromisoformat(r['ts'])
    eq_est = equity_at(ts) or float('nan')
    if eq_est and r.get('margin_usdt'):
        margin_pct = r['margin_usdt'] / eq_est * 100
    else:
        margin_pct = float('nan')
    size_rows.append({
        'ts': r['ts'][:16], 'asset': r['asset'], 'leverage': r['leverage'],
        'margin_usdt': r.get('margin_usdt', float('nan')),
        'est_equity': eq_est,
        'margin_pct_of_equity': round(margin_pct, 1) if pd.notna(margin_pct) else None,
        'position_size': r.get('position_size_usdt', float('nan')),
    })
size_df = pd.DataFrame(size_rows)
print(size_df.to_string(index=False))

valid = size_df.dropna(subset=['margin_pct_of_equity'])
if not valid.empty:
    print(f'\nmargin-as-%-of-equity stats:')
    print(f'  median: {valid["margin_pct_of_equity"].median():.1f}%')
    print(f'  range:  {valid["margin_pct_of_equity"].min():.1f}% → '
          f'{valid["margin_pct_of_equity"].max():.1f}%')

## 8. What's codifiable (bot-viable) vs. what isn't

The bootstrap style decomposes into pieces with very different replicability.

### Codifiable

| Element | Why it's codifiable |
|---|---|
| **OP-dominant asset choice** | Filter on high-vol mid-cap perps (rolling 30-day vol > X) |
| **20x default leverage** | Constant in config |
| **Step-up leverage with equity** | If equity > $1000, allow 50x. If > $2000, allow 100x. Rule-table |
| **Trade only on liquid pairs** | Filter by `quote_volume` over rolling window |
| **Cross margin mode** | Constant — same as he used in bootstrap |
| **Deep TP horizon** | Target = next major support (long) / resistance (short) |
| **Iron-hand on drawdown ≤ -10%** | Don't auto-cut on adverse excursion <X% |

### NOT codifiable (without infrastructure)

| Element | Why it's hard |
|---|---|
| **DCA-up on winners** | Needs partial-add infrastructure + dynamic margin sizing |
| **Same-day reversal** | Requires real-time invalidation detection |
| **'Sl manual' discretionary stops** | The whole point is NOT having a hard stop. Bots need one. |
| **Reading liquidation cascades** | Real-time aggregated-tape monitoring (aggr.trade) |
| **Asset rotation by 'feel'** | Discretionary — not encoded |
| **The 200x leverage "final-try" plays** | Risk-of-ruin too high without his discretion |

### The bootstrap math

If you compound a small account with a single 113% trade (his June 18 OP
short result), you're already at 2.13x. Three such trades = 9.7x. Five =
44x. That's the *theoretical* bootstrap path — and it works once or twice
by luck, then blows up (his Feb 2025 $18k → $0 incident confirms).

**A bot-viable bootstrap strategy doesn't replicate this**. It replicates
the *components* (asset selection, leverage scaling) without the
discretionary leg of conviction holds and DCA-into-loss.

### Recommendation for the repo

Don't build a bootstrap-clone sleeve. Instead:

1. Add an `op_perp_1m`-aware version of `chento_limit_bid_v1` that
   trades OP **using the same MTF cells we identified for BTC**. Bigger
   % moves per signal at the same R:R = faster compounding.
2. Add a per-sleeve **leverage step-up rule** keyed to equity tier:
   - Equity < $1k: 10x max
   - Equity $1k–$5k: 25x max
   - Equity > $5k: 50x max
   This mirrors his observed progression without trying to match his
   peak 125x bets.
3. Document the bootstrap *philosophy* in the README — it's about high-vol
   alts + asymmetric R:R + iron-hand on conviction. The bot version trades
   the structural signal at modest leverage.

## Summary

- **Bootstrap span**: June 9 2024 → November 7 2024 (~5 months)
- **Trajectory**: $200 → drawdown to $130 → $578 by day 5 → $4,124 by day 10 → multi-thousand by Sept 2024
- **Engine**: OPUSDT, 20-50x cross, deep TP targets (-20% to -47%), iron-hold through drawdowns
- **The killer move**: 2024-06-18 OP short, $461 margin → +$521 PnL (+113%) in one trade — verified by reading the position card AND cross-referencing with actual OPUSDT 1m bars (real spot move was -13.31% in 1 hour on a 31.7M-volume liquidation cascade)
- **Asset mix**: 79% OP, 21% BTC (no BTC trades until 2024-08-15, week 9 of the journey)
- **Leverage ramp**: 20x default → first 50x at week 5 → 75x at week 11 → 125x at week 16
- **Why he stopped (Nov 2024)**: "Not bringing me what it used to bring" — saturation in his community + tighter market regimes meant the same edge didn't compound as fast
- **What killed phase 2** (Feb 2025 $18k → $0): adding to losers without invalidation. He explicitly banned this in the Jan 2026 rulebook (*"adding to losers"* → forbidden)

The bootstrap phase taught him risk management by exposure. The mature
phase is more mechanical and less exciting but more durable.